In [61]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pdb # for debugging

import warnings
warnings.filterwarnings('ignore')


In [62]:
df = pd.read_csv('https://raw.githubusercontent.com/renatomaaliw3/public_files/refs/heads/master/Data%20Sets/Apriori%20Data.csv')
df.head(3)

,Technical Skills,Professional Certifications,Communication Skills,Grit,Work Habits,Employability
0,Technical Skills - Fair,Professional Certifications - 1,Communication Skills - 3,Grit - 1,Work Habits - 3,Less Employable
1,Technical Skills - Fair,Professional Certifications - 1,Communication Skills - 4,Grit - 2,Work Habits - 4,Employable
2,Technical Skills - Very Good,Professional Certifications - 2,Communication Skills - 5,Grit - 2,Work Habits - 4,Employable


In [63]:
# Data Preprocessing
# Before Applying the Apriori algorithm, we need to preprocess the data
# One-Hot Encoding, Remember get dummies? (but this is different)

from mlxtend.preprocessing import TransactionEncoder

# Consolidate each transaction into a single list of items, removing NaN values
transactions = df.apply(lambda row: row.dropna().tolist(), axis = 1).tolist()

# Initialize TransactionEncoder
encoder = TransactionEncoder()

# Fit and transform the transactions data
transaction_matrix = encoder.fit_transform(transactions)

# Convert to DataFrame
transaction_df = pd.DataFrame(transaction_matrix, columns = encoder.columns_)
transaction_df

,Communication Skills - 2,Communication Skills - 3,Communication Skills - 4,Communication Skills - 5,Employable,Grit - 1,Grit - 2,Less Employable,Professional Certifications - 1,Professional Certifications - 2,Technical Skills - Excellent,Technical Skills - Fair,Technical Skills - Good,Technical Skills - Pass,Technical Skills - Very Good,Underemployed,Work Habits - 3,Work Habits - 4,Work Habits - 5
0,False,True,False,False,False,True,False,True,True,False,False,True,False,False,False,False,True,False,False
1,False,False,True,False,True,False,True,False,True,False,False,True,False,False,False,False,False,True,False
2,False,False,False,True,True,False,True,False,False,True,False,False,False,False,True,False,False,True,False
3,False,False,True,False,True,False,True,False,False,True,False,True,False,False,False,False,True,False,False
4,False,False,True,False,True,False,True,False,True,False,False,False,True,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,False,False,False,True,True,False,True,False,False,True,False,True,False,False,False,False,False,False,True
496,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,True,True,False,False
497,False,False,True,False,True,False,True,False,False,True,True,False,False,False,False,False,True,False,False
498,False,False,False,True,False,True,False,False,True,False,False,False,True,False,False,True,False,True,False


In [64]:
# Appying the Apriori Algorithm
# Since data are cleaned and prepared for frequent itemset

from mlxtend.frequent_patterns import apriori, association_rules

# Apply the Apriori algorithm
frequent_itemsets = apriori(transaction_df, min_support = 0.2, use_colnames = True)

# min_support is the minimum support threshold. Itemsets with support greater than or equal to this threshold will be returned.
#use_colnames = True ensures that the item names are used in the output instead of column indices.


In [65]:
# View Frequent Itemsets

import warnings
warnings.filterwarnings('ignore', 'all')

frequent_itemsets

,support,itemsets
0,0.486,(Communication Skills - 4)
1,0.332,(Communication Skills - 5)
2,0.546,(Employable)
3,0.478,(Grit - 1)
4,0.522,(Grit - 2)
5,0.520,(Professional Certifications - 1)
6,0.480,(Professional Certifications - 2)
7,0.542,(Technical Skills - Fair)
8,0.352,(Technical Skills - Good)
9,0.294,(Underemployed)


In [66]:
# Generate Association Rules

pd.set_option('display.max_columns', 100)

rules = association_rules(frequent_itemsets, num_itemsets = len(transaction_df), metric = "confidence", min_threshold = 0.2)
rules.loc[:, :'lift']

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,(Communication Skills - 4),(Employable),0.486,0.546,0.268,0.551440,1.009964
1,(Employable),(Communication Skills - 4),0.546,0.486,0.268,0.490842,1.009964
2,(Communication Skills - 4),(Grit - 1),0.486,0.478,0.232,0.477366,0.998674
3,(Grit - 1),(Communication Skills - 4),0.478,0.486,0.232,0.485356,0.998674
4,(Grit - 2),(Communication Skills - 4),0.522,0.486,0.254,0.486590,1.001214
...,...,...,...,...,...,...,...
199,"(Professional Certifications - 1, Grit - 1)","(Technical Skills - Fair, Underemployed)",0.468,0.238,0.238,0.508547,2.136752
200,(Technical Skills - Fair),"(Grit - 1, Professional Certifications - 1, Un...",0.542,0.294,0.238,0.439114,1.493586
201,(Underemployed),"(Technical Skills - Fair, Professional Certifi...",0.294,0.392,0.238,0.809524,2.065112
202,(Professional Certifications - 1),"(Grit - 1, Technical Skills - Fair, Underemplo...",0.520,0.238,0.238,0.457692,1.923077


In [67]:
# Find the itemset(s) with the least support
least_support_itemsets = frequent_itemsets.sort_values(by='support', ascending=True).iloc[0]
display(least_support_itemsets.head())

,46
support,0.202
itemsets,"(Grit - 2, Work Habits - 4, Employable)"


In [68]:
# Filter rules where the consequent is 'Employable'
employable_rules = rules[rules['consequents'].apply(lambda x: 'Employable' in x)]

# Sort the rules by a metric like lift or confidence to find the 'most superior' rule
# Lift is a good indicator of the strength of the association
most_superior_rule = employable_rules.sort_values(by='lift', ascending=False).iloc[0]

# Display the most superior rule(s)
display(most_superior_rule.head())

,169
antecedents,"(Professional Certifications - 2, Grit - 2)"
consequents,"(Communication Skills - 5, Employable)"
antecedent support,0.47
consequent support,0.252
support,0.242


In [72]:
# Filter rules that contain both 'Underemployed' and 'Grit - 1' in either the antecedent or consequent
underemployed_grit_rules = rules[
    rules['antecedents'].apply(lambda x: 'Underemployed' in x or 'Grit - 1' in x) &
    rules['consequents'].apply(lambda x: 'Underemployed' in x or 'Grit - 1' in x)].iloc[0:1]

display(underemployed_grit_rules)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
30,(Grit - 1),(Underemployed),0.478,0.294,0.294,0.615063,2.09205,1.0,0.153468,1.834065,1.0,0.615063,0.454763,0.807531


In [71]:
# Filter the rules to find the one where the antecedent is {'Grit - 2', 'Technical Skills - Good'} and the consequent is {'Employable'}
specific_rule = rules[
    rules['antecedents'].apply(lambda x: set(x) == {'Grit - 2', 'Technical Skills - Good'}) &
    rules['consequents'].apply(lambda x: set(x) == {'Employable'})
]

# Display the specific rule
display(specific_rule)

# Explain the association based on the rule's metrics
if not specific_rule.empty:
    confidence = specific_rule['confidence'].values[0]
    lift = specific_rule['lift'].values[0]
    print(f"\nBased on the association rule:")
    print(f"- Confidence: {confidence:.4f}")
    print(f"- Lift: {lift:.4f}")

    if confidence > 0.5:
        print("\nThis rule has a confidence greater than 0.5, suggesting that when an applicant has both 'Grit - 2' and 'Technical Skills - Good', it is likely that they are 'Employable'.")
    else:
        print("\nThis rule has a confidence less than or equal to 0.5, suggesting that having both 'Grit - 2' and 'Technical Skills - Good' doesn't strongly guarantee that an applicant is 'Employable'.")

    if lift > 1:
        print("The lift value is greater than 1, indicating that the presence of {'Grit - 2', 'Technical Skills - Good'} in a transaction increases the likelihood of {'Employable'} appearing in the same transaction, more than would be expected by chance.")
    elif lift == 1:
        print("The lift value is equal to 1, suggesting that there is no association between {'Grit - 2', 'Technical Skills - Good'} and {'Employable'}.")
    else:
        print("The lift value is less than 1, suggesting that the presence of {'Grit - 2', 'Technical Skills - Good'} in a transaction decreases the likelihood of {'Employable'} appearing in the same transaction.")

else:
    print("No association rule found with {'Grit - 2', 'Technical Skills - Good'} as the antecedent and {'Employable'} as the consequent.")

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
100,"(Technical Skills - Good, Grit - 2)",(Employable),0.292,0.546,0.292,1.0,1.831502,1.0,0.132568,inf,0.641243,0.534799,1.0,0.767399



Based on the association rule:
- Confidence: 1.0000
- Lift: 1.8315

This rule has a confidence greater than 0.5, suggesting that when an applicant has both 'Grit - 2' and 'Technical Skills - Good', it is likely that they are 'Employable'.
The lift value is greater than 1, indicating that the presence of {'Grit - 2', 'Technical Skills - Good'} in a transaction increases the likelihood of {'Employable'} appearing in the same transaction, more than would be expected by chance.
